# X5 · Uplift baseline and treatment-assignment diagnostics

**Session 7 · X5 Retail Growth Experimentation & ML Decisioning Platform**

**Research question:** Do treatment and control customers look comparable on observable historical characteristics, and can a simple T-learner produce a useful *predicted uplift* ranking?

**Data:** The dbt-built `RETAIL_GROWTH.DEV_MARTS.MART_UPLIFT_TRAINING` table (200,039 labeled customers), temporarily cached to an untracked local Parquet file. The upstream feature pipeline excludes treatment and outcome from predictors and uses a historical cutoff.

**Interpretation boundary:** Public X5 documentation does not establish random assignment. Differences between treated and control outcomes, IPW-adjusted differences, and uplift-decile contrasts are **associational estimates unless additional causal identification assumptions hold**. A propensity AUC near 0.50 does not prove randomization.

Run cells **top to bottom** from `notebooks/` with the project's `project_env` kernel. The `.env` file stays outside Git; the Snowflake password is requested at runtime only if the cache does not exist. Code is preserved from the supplied notebook script; the new material is explanatory Markdown and comments.

## 1 · Imports

Load the numerical, plotting, Snowflake, and scikit-learn components. `clone` gives each outcome model an independent copy of its preprocessing pipeline; it matters when models are fitted to different treatment arms.

In [ ]:
# Dependencies for data access, diagnostics, and the two-model T-learner.

from pathlib import Path
from getpass import getpass
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import snowflake.connector

from dotenv import load_dotenv

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

## 2 · Project paths and local cache

Resolve the project root relative to the `notebooks/` working directory, load account/user variables from a Git-ignored `.env`, and define a Git-ignored Parquet cache. The cache is a modeling convenience, **not a second source of truth**.

In [ ]:
# Resolve paths once and create only the local cache directory.
# No account credentials or raw data are written into the notebook.

PROJECT_ROOT = Path("..").resolve()

load_dotenv(
    PROJECT_ROOT / ".env"
)

DATA_DIR = PROJECT_ROOT / "data"

CACHE_PATH = (
    DATA_DIR
    / "processed"
    / "mart_uplift_training.parquet"
)

CACHE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

## 3 · Load Snowflake data once

If the Parquet extract exists, use it. Otherwise, read the existing dbt mart from Snowflake, lowercase column names, and cache the table locally. `getpass` avoids embedding the Snowflake password in source code. If the mart changes, delete/refresh the cache intentionally.

In [ ]:
# Read the model-ready dbt mart; do NOT recreate customer features in pandas.
# Close Snowflake resources even if a fetch fails.

if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)

else:

    connection = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        password=getpass("Snowflake password: "),
        authenticator="username_password_mfa",
        role="DBT_DEV_ROLE",
        warehouse="RETAIL_DEV_WH",
        database="RETAIL_GROWTH",
        schema="DEV_MARTS",
    )

    cursor = connection.cursor()

    try:
        cursor.execute(
            """
            SELECT *
            FROM RETAIL_GROWTH.DEV_MARTS.MART_UPLIFT_TRAINING
            """
        )

        df = cursor.fetch_pandas_all()

    finally:
        cursor.close()
        connection.close()

    df.columns = df.columns.str.lower()

    df.to_parquet(
        CACHE_PATH,
        index=False,
    )

df.columns = df.columns.str.lower()

## 4 · Inspect modeling identifiers and labels

Preview the customer identifier, treatment indicator, and observed target. `client_id` is used for identity and reconciliation, not as a predictive feature.

In [ ]:
# Check the three fields that define the labeled modeling population.

df[[
    "client_id",
    "treatment_flg",
    "target",
]].head()

## 5 · Assert the modeling population

These checks protect against accidentally fitting on the wrong table or an incomplete extract. They are a local modeling checkpoint; dbt tests remain the upstream data contracts.

In [ ]:
# Fail early if the extract has the wrong number of rows or labels.

assert len(df) == 200_039
assert df["client_id"].nunique() == 200_039
assert set(df["treatment_flg"].unique()) == {0, 1}
assert set(df["target"].unique()) == {0, 1}

## 6 · Unadjusted treatment and control outcomes

Group customers by **observed treatment assignment** and compute the binary target rate in each group. This is the descriptive contrast that later adjustment and uplift models are compared against.

In [ ]:
# Within a binary group, the mean of target equals its observed target rate.

outcome_summary = (
    df.groupby("treatment_flg")["target"]
    .agg(
        customers="size",
        target_count="sum",
        target_rate="mean",
    )
)

outcome_summary

## 7 · Absolute and relative differences

The **absolute** difference is treated rate minus control rate and is reported in percentage points. The **relative** difference divides that change by the control rate. Neither number alone identifies a causal effect without treatment-assignment assumptions.

In [ ]:
# Distinguish percentage points from relative percent change.

control_rate = outcome_summary.loc[0, "target_rate"]
treatment_rate = outcome_summary.loc[1, "target_rate"]

observed_difference = (
    treatment_rate - control_rate
)

relative_difference = (
    observed_difference / control_rate
)

print(
    f"Control rate: {control_rate:.4%}"
)

print(
    f"Treatment rate: {treatment_rate:.4%}"
)

print(
    f"Observed difference: "
    f"{observed_difference:.4%}"
)

print(
    f"Relative difference: "
    f"{relative_difference:.2%}"
)

## 8 · Balance for categorical gender

Numeric SMDs do not summarize a string-valued field such as `gender`. Compare the fraction of F/M/U **within each treatment arm** with a column-normalized cross-tab; these percentages are proportions (0–1), despite the `_pct` column names.

In [ ]:
# Normalize within treatment arm so each column sums to one.

gender_balance = (
    pd.crosstab(
        df["gender"],
        df["treatment_flg"],
        normalize="columns",
    )
    .rename(
        columns={
            0: "control_pct",
            1: "treatment_pct",
        }
    )
)

gender_balance

## 9 · Declare predictors and exclusions

Exclude IDs, treatment, and outcome from the feature matrix. The gender column is categorical; remaining modeled columns are numeric. These features were built from the historical pre-communication feature layer.

In [ ]:
# Never permit treatment_flg or target to enter the feature matrix.

EXCLUDE_COLUMNS = {
    "client_id",
    "treatment_flg",
    "target",
}

feature_columns = [
    column
    for column in df.columns
    if column not in EXCLUDE_COLUMNS
]

categorical_features = [
    "gender",
]

numeric_features = [
    column
    for column in feature_columns
    if column not in categorical_features
]

## 10 · Standardized mean difference (SMD)

SMD expresses an observed treated-control mean difference in pooled standard-deviation units, letting us compare age, spend, loyalty, etc. on a common scale. This function handles constant pooled variance by returning zero; it does not separately quantify missingness balance.

In [ ]:
# Numerator = treated mean minus control mean.
# Denominator = square root of the average group variance.

def standardized_mean_difference(
    treatment_values,
    control_values,
):
    treatment_values = pd.to_numeric(
        treatment_values,
        errors="coerce",
    )

    control_values = pd.to_numeric(
        control_values,
        errors="coerce",
    )

    treatment_mean = treatment_values.mean()
    control_mean = control_values.mean()

    treatment_var = treatment_values.var()
    control_var = control_values.var()

    pooled_sd = np.sqrt(
        (
            treatment_var
            +
            control_var
        )
        / 2
    )

    if pooled_sd == 0:
        return 0.0

    return (
        treatment_mean
        -
        control_mean
    ) / pooled_sd

## 11 · Audit numeric covariate balance

Compute the signed SMD and its absolute magnitude for every numeric feature; sort by absolute value to identify the most different observed pre-treatment variables. A commonly used |SMD|=0.10 line is a heuristic, **not proof of randomization or lack of confounding**.

In [ ]:
# Use all labeled customers for these descriptive assignment diagnostics.

balance_rows = []

treated = df["treatment_flg"] == 1
control = df["treatment_flg"] == 0

for feature in numeric_features:

    smd = standardized_mean_difference(
        df.loc[treated, feature],
        df.loc[control, feature],
    )

    balance_rows.append(
        {
            "feature": feature,
            "smd": smd,
            "abs_smd": abs(smd),
        }
    )

balance = (
    pd.DataFrame(balance_rows)
    .sort_values(
        "abs_smd",
        ascending=False,
    )
)

balance.head(15)

## 12 · Visual balance check

Plot the 20 largest absolute SMDs. The vertical line at 0.10 is a visual reference, not a formal hypothesis test or automatic pass/fail decision.

In [ ]:
# Show the largest imbalances in a common standardized unit.

plot_balance = (
    balance
    .head(20)
    .sort_values("abs_smd")
)

plt.figure(figsize=(8, 7))

plt.barh(
    plot_balance["feature"],
    plot_balance["abs_smd"],
)

plt.axvline(
    0.10,
    linestyle="--",
)

plt.xlabel("Absolute standardized mean difference")
plt.ylabel("Feature")
plt.title("Largest observed treatment-control imbalances")

plt.tight_layout()
plt.show()

## 13 · Treatment-propensity preprocessing

The propensity score is `P(treatment=1 | historical X)`. Missing numeric values receive training-fold medians and are standardized; gender receives training-fold mode imputation and one-hot encoding. Wrapping preprocessing in a pipeline prevents fitted transforms from leaking across cross-validation folds.

In [ ]:
# Numeric: impute then scale. Categorical: impute then encode.
# The pipeline fits these steps inside each training fold.

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

# Encode gender separately because a string category is not a numeric magnitude.
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

# Combine numeric and categorical transformations into a single fitted pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features,
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features,
        ),
    ]
)

# Model only treatment assignment here, not the purchase target.
propensity_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000
            )
        ),
    ]
)

## 14 · Out-of-fold assignment predictions

Five stratified folds produce propensity predictions for customers **not used to fit their own fold model**. ROC-AUC tests how predictable assignment is from observed covariates; ~0.50 is weak predictability, not evidence that assignment was actually randomized.

In [ ]:
# cross_val_predict refits inside each fold, then concatenates held-out probabilities.
# These scores are used for diagnostics and IPW, NOT as an uplift-model input.

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

X = df[feature_columns]

treatment = df["treatment_flg"]

propensity_scores = cross_val_predict(
    propensity_model,
    X,
    treatment,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

propensity_auc = roc_auc_score(
    treatment,
    propensity_scores,
)

propensity_auc

## 15 · Check overlap numerically

Compare the distribution of estimated treatment probabilities between treated and control customers. Similar distributions support overlap for measured covariates; extreme scores near zero or one can destabilize weighting.

In [ ]:
# Append propensity_score only to the diagnostic working dataframe.

df["propensity_score"] = propensity_scores

df.groupby(
    "treatment_flg"
)["propensity_score"].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
)

## 16 · Visualize common support

Overlaid, density-normalized histograms reveal whether comparable propensity-score ranges occur in both groups. This checks **estimated** overlap, not unobserved confounding.

In [ ]:
# density=True compares distribution shapes even with unequal group counts.

plt.figure(figsize=(8, 5))

plt.hist(
    df.loc[
        df["treatment_flg"] == 0,
        "propensity_score",
    ],
    bins=50,
    alpha=0.5,
    density=True,
    label="Control",
)

plt.hist(
    df.loc[
        df["treatment_flg"] == 1,
        "propensity_score",
    ],
    bins=50,
    alpha=0.5,
    density=True,
    label="Treatment",
)

plt.xlabel("Estimated propensity score")
plt.ylabel("Density")
plt.title("Treatment propensity overlap")
plt.legend()

plt.tight_layout()
plt.show()

## 17 · Stabilized inverse-probability weights

Under exchangeability on measured X, positivity, consistency, and suitable propensity specification, inverse-probability weighting targets a balanced pseudo-population. We use the observed treatment fraction in the numerator and clip very extreme estimated probabilities to 0.01–0.99 to limit variance; clipping changes the estimand in extreme cases.

In [ ]:
# Treated: P(T=1)/e(X). Control: P(T=0)/(1-e(X)).
# Check the size and stability of weights before interpretation.

EPSILON = 0.01

propensity_clipped = np.clip(
    propensity_scores,
    EPSILON,
    1 - EPSILON,
)

treatment_probability = treatment.mean()

weights = np.where(
    treatment == 1,

    treatment_probability
    / propensity_clipped,

    (1 - treatment_probability)
    / (1 - propensity_clipped),
)

## 18 · Inspect weight stability

Inspect median, upper percentiles, and maximum. A few huge weights would indicate unstable adjustment or poor overlap even if the mean is near one.

In [ ]:
# A diagnostic only: do not interpret a mean weight near one as proof of unbiasedness.

pd.Series(weights).describe(
    percentiles=[
        0.01,
        0.05,
        0.50,
        0.95,
        0.99,
    ]
)

## 19 · Compare unadjusted and IPW-adjusted rates

Calculate outcome rates separately inside treated and control arms using their stabilized weights, then subtract. This is a weighted **group contrast**; the notebook does not estimate a confidence interval or demonstrate that unmeasured confounding is absent.

In [ ]:
# np.average(value, weights=...) returns each arm’s normalized weighted rate.
# Report raw and IPW differences side by side.

target = df["target"].to_numpy()

treatment_array = treatment.to_numpy()

weighted_treatment_rate = np.average(
    target[treatment_array == 1],
    weights=weights[
        treatment_array == 1
    ],
)

weighted_control_rate = np.average(
    target[treatment_array == 0],
    weights=weights[
        treatment_array == 0
    ],
)

ipw_difference = (
    weighted_treatment_rate
    -
    weighted_control_rate
)

print(
    f"Raw observed difference: "
    f"{observed_difference:.4%}"
)

print(
    f"IPW-adjusted difference: "
    f"{ipw_difference:.4%}"
)

## 20 · Freeze uplift training/validation split

Remove the diagnostic propensity column, reconstruct predictor/label arrays, and stratify the split on the four treatment × outcome combinations. The held-out 25% is **development validation**; later model comparison used it, so it is not an untouched final test set.

In [ ]:
# Use exactly the same random seed and stratification in Session 8.
# Keep diagnostic propensity_score out of both T-learner outcomes.

model_df = df.drop(
    columns=["propensity_score"]
)

X = model_df[feature_columns]

treatment = model_df["treatment_flg"]

target = model_df["target"]

strata = (
    treatment.astype(str)
    + "_"
    + target.astype(str)
)

train_index, validation_index = train_test_split(
    np.arange(len(model_df)),
    test_size=0.25,
    random_state=42,
    stratify=strata,
)

## 21 · Independent outcome-model factory

A T-learner fits separate response models for treated and control customers. `clone(preprocessor)` avoids reusing one already-fitted transformer object between those independent models.

In [ ]:
# A fresh preprocessing pipeline and logistic regression are created per arm.

def make_outcome_model():

    return Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000
                ),
            ),
        ]
    )

## 22 · Fit the treated and control models

Filter **within the training index** before fitting each model. Neither response model sees validation labels. Each estimator learns an observed outcome probability in its own arm.

In [ ]:
# Train treated model only on T=1; control model only on T=0.

treatment_model = make_outcome_model()
control_model = make_outcome_model()

treatment_train_mask = (
    treatment.iloc[train_index]
    .to_numpy()
    == 1
)

control_train_mask = (
    treatment.iloc[train_index]
    .to_numpy()
    == 0
)

treatment_model.fit(
    X.iloc[train_index][
        treatment_train_mask
    ],
    target.iloc[train_index][
        treatment_train_mask
    ],
)

# Fit control on a DISJOINT subset with its own fitted preprocessing.
control_model.fit(
    X.iloc[train_index][
        control_train_mask
    ],
    target.iloc[train_index][
        control_train_mask
    ],
)

## 23 · Predict both potential response probabilities

For each *same* validation customer, obtain a prediction from each arm-specific model and subtract: `predicted_uplift = p_treatment - p_control`. It is a **model-based contrast**, not an observed individual causal effect.

In [ ]:
# predict_proba(... )[:, 1] selects the probability of target=1.

X_validation = X.iloc[
    validation_index
]

p_treatment = (
    treatment_model.predict_proba(
        X_validation
    )[:, 1]
)

# Predict the untreated response for exactly the same held-out rows.
p_control = (
    control_model.predict_proba(
        X_validation
    )[:, 1]
)

predicted_uplift = (
    p_treatment
    -
    p_control
)

## 24 · Assemble the held-out evaluation frame

Store identifiers, actual arm and target, both modeled probabilities, and the predicted difference together. Keeping this table aligned lets all subsequent diagnostics compare the same customers.

In [ ]:
# One row per held-out customer; do not treat both predicted outcomes as observed.

validation_results = pd.DataFrame(
    {
        "client_id": model_df.iloc[
            validation_index
        ]["client_id"].to_numpy(),

        "treatment_flg": treatment.iloc[
            validation_index
        ].to_numpy(),

        "target": target.iloc[
            validation_index
        ].to_numpy(),

        "p_treatment": p_treatment,

        "p_control": p_control,

        "predicted_uplift": predicted_uplift,
    }
)

## 25 · Inspect predicted uplift spread

Look at mean, median, tails, and negative scores. Wide tails can indicate meaningful heterogeneity **or** model uncertainty; they are not automatically evidence of treatment harm/benefit for specific individuals.

In [ ]:
# The uplift column is a probability difference, not a percentage already.

validation_results[
    "predicted_uplift"
].describe(
    percentiles=[
        0.01,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.99,
    ]
)

## 26 · Response-model diagnostics by observed arm

ROC-AUC checks discrimination and Brier score checks squared probability error. Evaluate `p_control` **only among observed control outcomes** and `p_treatment` **only among observed treated outcomes**. These scores do not directly measure uplift-ranking quality.

In [ ]:
# Each potential-outcome model is scored only where that outcome was observed.

control_validation = (
    validation_results["treatment_flg"]
    == 0
)

control_auc = roc_auc_score(
    validation_results.loc[
        control_validation,
        "target",
    ],
    validation_results.loc[
        control_validation,
        "p_control",
    ],
)

control_brier = brier_score_loss(
    validation_results.loc[
        control_validation,
        "target",
    ],
    validation_results.loc[
        control_validation,
        "p_control",
    ],
)

treated_validation = (
    validation_results["treatment_flg"]
    == 1
)

treatment_auc = roc_auc_score(
    validation_results.loc[
        treated_validation,
        "target",
    ],
    validation_results.loc[
        treated_validation,
        "p_treatment",
    ],
)

treatment_brier = brier_score_loss(
    validation_results.loc[
        treated_validation,
        "target",
    ],
    validation_results.loc[
        treated_validation,
        "p_treatment",
    ],
)

## 27 · Summarize outcome-model performance

Place treatment and control ROC-AUC/Brier values in one compact comparison. Session 8 tests whether stronger outcome prediction actually translates to stronger uplift ranking.

In [ ]:
# Higher AUC and lower Brier are favorable for OUTCOME prediction, not sufficient for uplift.

pd.DataFrame(
    {
        "group": [
            "Treatment model",
            "Control model",
        ],
        "roc_auc": [
            treatment_auc,
            control_auc,
        ],
        "brier_score": [
            treatment_brier,
            control_brier,
        ],
    }
)

## 28 · Examine highest predicted scores

Inspect the highest modeled contrasts as a debugging check; an actual `target=0` for a highly ranked treated customer does not invalidate an average-probability prediction.

In [ ]:
# Show examples; never claim an individual outcome was caused by treatment.

validation_results.sort_values(
    "predicted_uplift",
    ascending=False,
).head(10)

## 29 · Examine lowest predicted scores

Likewise, low or negative predicted differences represent the model’s hypothesis. We cannot directly observe both potential outcomes for an individual.

In [ ]:
# Inspect negative-tail cases as model diagnostics, not proven harm.

validation_results.sort_values(
    "predicted_uplift",
).head(10)

## 30 · Preliminary uplift deciles

`pd.qcut` forms approximately equal-sized bins of predicted uplift; **decile 0 is lowest, decile 9 highest**. Within each bin, compute observed target rates separately by treatment arm. These are exploratory comparisons, not yet formal Qini/AUUC metrics.

In [ ]:
# Equal-frequency bins summarize how observed outcomes vary across predicted ranks.

validation_results[
    "uplift_decile"
] = pd.qcut(
    validation_results[
        "predicted_uplift"
    ],
    q=10,
    labels=False,
    duplicates="drop",
)

# Summarize actual outcomes within each predicted-uplift bucket and arm.
decile_summary = (
    validation_results
    .groupby(
        [
            "uplift_decile",
            "treatment_flg",
        ]
    )
    ["target"]
    .agg(
        customers="size",
        target_rate="mean",
    )
    .reset_index()
)

decile_summary

## 31 · Compare deciles side by side

Pivot the two arm rates into columns, subtract them, and sort highest predicted-uplift decile first. Sampling variation means observed uplift need not decrease perfectly from decile 9 to 0.

In [ ]:
# observed_uplift here is a within-decile treated-control rate contrast.

decile_uplift = (
    decile_summary
    .pivot(
        index="uplift_decile",
        columns="treatment_flg",
        values="target_rate",
    )
    .rename(
        columns={
            0: "control_rate",
            1: "treatment_rate",
        }
    )
)

decile_uplift["observed_uplift"] = (
    decile_uplift["treatment_rate"]
    -
    decile_uplift["control_rate"]
)

decile_uplift = (
    decile_uplift
    .sort_index(ascending=False)
)

decile_uplift

---

## Conclusions and limitations

**What the notebook establishes:** observed pretreatment covariate balance, out-of-fold treatment-assignment predictability, estimated propensity overlap, IPW sensitivity to *measured* covariates, and an interpretable logistic T-learner baseline. On the results previously reviewed, the propensity ROC-AUC was about 0.499, and adjustment changed the raw 3.3231-percentage-point gap to about 3.2612 percentage points. Those are **descriptive and model-dependent findings**.

**What it does not establish:** that X5 assignment was randomized, that no unmeasured confounding exists, or that any individual customer's two potential outcomes can both be observed.

**Next step:** recreate the identical development split, evaluate ranking with uplift@30%, a clearly defined Qini-style gain, and AUUC, and compare/tune an additional T-learner.